In [14]:
import copy
from heapq import heappush, heappop

n = 4
# Hướng đi: Lên, Xuống, Trái, Phải
rows = [-1, 1, 0, 0]
cols = [0, 0, -1, 1]
ten_huong = ["Lên", "Xuống", "Trái", "Phải"]

class priorityQueue:
    def __init__(self):
        self.heap = []
    def push(self, k):
        heappush(self.heap, k)
    def pop(self):
        return heappop(self.heap)
    def empty(self):
        return not self.heap

class node:
    def __init__(self, parent, mats, empty_tile_posi, costs, levels):
        self.parent = parent
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.costs = costs
        self.levels = levels

    def __lt__(self, nxt):
        return self.costs < nxt.costs

# ---------------------------------------------------------
# 1. DEBUG HÀM TÍNH CHI PHÍ (SỐ Ô SAI VỊ TRÍ)
# ---------------------------------------------------------
def calculateCost(mats, final) -> int:
    count = 0
    sai_vi_tri = []
    for i in range(n):
        for j in range(n):
            if mats[i][j] and (mats[i][j] != final[i][j]):
                count += 1
                sai_vi_tri.append(mats[i][j]) # Lưu lại các số bị sai để in debug

    print(f"      [Hàm calculateCost] Đếm được {count} ô sai vị trí (Các số sai: {sai_vi_tri})")
    return count

# ---------------------------------------------------------
# 2. DEBUG HÀM TẠO NODE MỚI (TRẠNG THÁI MỚI)
# ---------------------------------------------------------
def newNode(mats, empty_tile_posi, new_empty_tile_posi, levels, parent, final) -> node:
    print(f"    [Hàm newNode] Bắt đầu tạo Node mới (Level: {levels})...")
    new_mats = copy.deepcopy(mats)
    x1, y1 = empty_tile_posi
    x2, y2 = new_empty_tile_posi

    so_bi_day = new_mats[x2][y2]
    print(f"      -> Đổi chỗ ô trống (0) với số {so_bi_day} (từ tọa độ {new_empty_tile_posi} sang {empty_tile_posi})")

    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    costs = calculateCost(new_mats, final)
    print(f"    [Hàm newNode] Tạo thành công Node! Trạng thái mới có Cost = {costs}")
    return node(parent, new_mats, new_empty_tile_posi, costs, levels)

def printMatrix(mats):
    for i in range(n):
        for j in range(n):
            print("%2d " %(mats[i][j]), end=" ")
        print()

def isSafe(x, y):
    return 0 <= x < n and 0 <= y < n

def printPath(root):
    if root is None:
        return
    printPath(root.parent)
    printMatrix(root.mats)
    print("------------------")

# ---------------------------------------------------------
# 3. DEBUG HÀM DUYỆT CHÍNH
# ---------------------------------------------------------
def solve(initial, empty_tile_pos, final):
    print("=== KHỞI TẠO BÀI TOÁN ===")
    pq = priorityQueue()
    print("[Hàm solve] Đang tính cost cho trạng thái ban đầu:")
    costs = calculateCost(initial, final)
    root = node(None, initial, empty_tile_pos, costs, 0)
    pq.push(root)

    step = 0
    while not pq.empty():
        step += 1
        print(f"\n================ VÒNG LẶP THỨ {step} ================")

        # Lấy node có cost nhỏ nhất ra khỏi hàng đợi
        minimum = pq.pop()

        print(f"[Hàm solve] Bốc Node TỐT NHẤT từ hàng đợi ra để duyệt:")
        print(f"   + Vị trí ô trống hiện tại: {minimum.empty_tile_posi}")
        print(f"   + Số bước đã đi (levels): {minimum.levels}")
        print(f"   + Chi phí (costs): {minimum.costs}")
        printMatrix(minimum.mats)

        # Điều kiện dừng
        if minimum.costs == 0:
            print("\n!!! TÌM THẤY LỜI GIẢI (Cost = 0) !!!")
            print("ĐƯỜNG ĐI CHI TIẾT TỪ ĐẦU ĐẾN ĐÍCH:")
            printPath(minimum)
            return

        print("\n[Hàm solve] Kiểm tra 4 hướng đi từ vị trí ô trống:")
        for i in range(n):
            new_x = minimum.empty_tile_posi[0] + rows[i]
            new_y = minimum.empty_tile_posi[1] + cols[i]

            huong = ten_huong[i]

            if isSafe(new_x, new_y):
                print(f"  * Hướng [{huong}] hợp lệ (Tọa độ mới: [{new_x}, {new_y}]). Gọi hàm newNode:")
                child = newNode(minimum.mats, minimum.empty_tile_posi,
                                [new_x, new_y], minimum.levels + 1, minimum, final)
                pq.push(child)
            else:
                print(f"  * Hướng [{huong}] bị cản (Tọa độ [{new_x}, {new_y}] ra ngoài ma trận). BỎ QUA.")


# Dữ liệu chạy thử
initial = [
    [1, 2, 3, 4],
    [5, 6, 0, 8],
    [9, 10, 7, 11],
    [13, 14, 15, 12]
]

final = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 0]
]

empty_tile_pos = [1, 2] # Ô trống (0) đang ở hàng 1, cột 2
solve(initial, empty_tile_pos, final)

=== KHỞI TẠO BÀI TOÁN ===
[Hàm solve] Đang tính cost cho trạng thái ban đầu:
      [Hàm calculateCost] Đếm được 3 ô sai vị trí (Các số sai: [7, 11, 12])

================ VÒNG LẶP THỨ 1 ================
[Hàm solve] Bốc Node TỐT NHẤT từ hàng đợi ra để duyệt:
   + Vị trí ô trống hiện tại: [1, 2]
   + Số bước đã đi (levels): 0
   + Chi phí (costs): 3
 1   2   3   4  
 5   6   0   8  
 9  10   7  11  
13  14  15  12  

[Hàm solve] Kiểm tra 4 hướng đi từ vị trí ô trống:
  * Hướng [Lên] hợp lệ (Tọa độ mới: [0, 2]). Gọi hàm newNode:
    [Hàm newNode] Bắt đầu tạo Node mới (Level: 1)...
      -> Đổi chỗ ô trống (0) với số 3 (từ tọa độ [0, 2] sang [1, 2])
      [Hàm calculateCost] Đếm được 4 ô sai vị trí (Các số sai: [3, 7, 11, 12])
    [Hàm newNode] Tạo thành công Node! Trạng thái mới có Cost = 4
  * Hướng [Xuống] hợp lệ (Tọa độ mới: [2, 2]). Gọi hàm newNode:
    [Hàm newNode] Bắt đầu tạo Node mới (Level: 1)...
      -> Đổi chỗ ô trống (0) với số 7 (từ tọa độ [2, 2] sang [1, 2])
      [Hàm calc